# Stage 30 — Extração de características PSD

Objetivo: transformar cada época filtrada em um vetor de potência espectral. Com 128 canais e duas bandas, cada época terá 256 características.


## 1. Importações


In [ ]:
from pathlib import Path

import numpy as np
from scipy.signal import periodogram


## 2. Caminhos e parâmetros


In [ ]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_20_band_filtering_and_epoch"
OUTPUT_DIR = PROJECT_ROOT / "processed_data" / "stage_30_feature_extraction_psd"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLING_RATE = 256
BANDS = [(8, 12), (12, 30)]

print("Entrada:", INPUT_DIR)
print("Saída:", OUTPUT_DIR)


## 3. Descobrir os arquivos produzidos pela Stage 20


In [ ]:
data_files = sorted(INPUT_DIR.glob("*_inner_bands.npy"))
print(f"Sessões encontradas: {len(data_files)}")
print("Exemplo:", data_files[0].name)


## 4. Carregar e inspecionar uma sessão

O eixo 0 são épocas, o eixo 1 são bandas, o eixo 2 são canais e o último eixo é o tempo.


In [ ]:
example_data = np.load(data_files[0])
print("Formato:", example_data.shape)
print("Tipo:", example_data.dtype)
print("Todos os valores são finitos:", np.isfinite(example_data).all())


## 5. Função de extração

O periodograma produz uma potência para cada frequência. Para cada banda, aplicamos uma máscara e somamos a potência ao longo das frequências.


In [ ]:
def extract_psd_features(data):
    frequencies, psd = periodogram(data, fs=SAMPLING_RATE, axis=-1)
    features_by_band = []

    for band_index, (low_frequency, high_frequency) in enumerate(BANDS):
        frequency_mask = (
            (frequencies >= low_frequency)
            & (frequencies <= high_frequency)
        )
        band_power = psd[:, band_index, :, frequency_mask].sum(axis=-1)
        features_by_band.append(band_power)

    return np.concatenate(features_by_band, axis=1)


## 6. Testar a extração em uma sessão


In [ ]:
example_features = extract_psd_features(example_data)
print("Entrada:", example_data.shape)
print("Saída (épocas, características):", example_features.shape)
print("Mínimo e máximo:", example_features.min(), example_features.max())


## 7. Processar e salvar todas as sessões


In [ ]:
for data_path in data_files:
    session_name = data_path.name.replace("_inner_bands.npy", "")
    labels_path = INPUT_DIR / f"{session_name}_inner_bands_labels.npy"

    data = np.load(data_path)
    labels = np.load(labels_path)
    features = extract_psd_features(data)

    if len(features) != len(labels):
        raise ValueError(f"Épocas e rótulos incompatíveis em {session_name}")

    np.save(OUTPUT_DIR / f"{session_name}_features_psd.npy", features)
    np.save(OUTPUT_DIR / f"{session_name}_labels.npy", labels)
    print(f"{session_name}: {features.shape}")

print("Stage 30 finalizada.")
